# Plant Disease Classification: Google Colab training

Trains the **same MobileNetV2 transfer-learning model** as the local Windows workflow. The source files embedded below are generated from the project repository (`scripts/build_notebook.py`), so preprocessing and architecture are identical.

**Steps**
1. `Runtime > Change runtime type > T4 GPU` (recommended).
2. Run the cells from top to bottom. Nothing is faked: if a step fails, the notebook stops with the real error.
3. At the end, download `plant_disease_model.keras`, `class_names.json` and `model_info.json` and put them in the web app's `models/` folder.

Dataset: PlantVillage (Hughes & Salathe 2015; Mohanty, Hughes & Salathe 2016), downloaded from GitHub or supplied as a ZIP in your Google Drive. Check the dataset's terms before redistributing it.

In [ ]:
import platform, sys
import tensorflow as tf
print("Python", platform.python_version(), "| TensorFlow", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU devices:", gpus if gpus else "NONE - training will be very slow. Switch the runtime to GPU.")

## 1. Settings

In [ ]:
# ---- edit these if you like ------------------------------------------------
USE_DRIVE = True                      # save the finished model files to Google Drive too
DRIVE_DIR = "/content/drive/MyDrive/plant_disease_project"
DATASET_SOURCE = "auto"               # "auto" = download from GitHub, "drive_zip" = use DRIVE_ZIP
DRIVE_ZIP = DRIVE_DIR + "/PlantVillage.zip"
EPOCHS = 10                           # head-training epochs (early stopping may stop sooner)
FINE_TUNE_EPOCHS = 5                  # 0 disables fine-tuning
BATCH_SIZE = 32
# -----------------------------------------------------------------------------
PROJECT = "/content/pdc"
OUT = "/content/output"               # model + results are written here
import os, subprocess, sys
for d in (PROJECT + "/src", PROJECT + "/training", PROJECT + "/dataset", OUT + "/models", OUT + "/results"):
    os.makedirs(d, exist_ok=True)
for init in (PROJECT + "/src/__init__.py", PROJECT + "/training/__init__.py"):
    open(init, "a").close()

def run(cmd):
    """Run a command and stream its output; raise if it fails."""
    print("$", " ".join(cmd))
    p = subprocess.Popen(cmd, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError("Command failed with exit code %d" % p.returncode)

In [ ]:
if USE_DRIVE or DATASET_SOURCE == "drive_zip":
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)

## 2. Project source files (generated from the repository; do not edit)

In [ ]:
%%writefile /content/pdc/src/constants.py
"""Constants shared by training, evaluation and inference.

Keeping them in one dependency-free module guarantees that the local scripts, the
Google Colab notebook and the Flask app all use the same preprocessing/architecture.
"""

IMG_SIZE = 224
IMG_SHAPE = (IMG_SIZE, IMG_SIZE, 3)

# Identifies the exact preprocessing contract. The saved model takes RGB pixels in the
# range 0..255 (float32); resizing happens in preprocessing.py and the rescale to
# [-1, 1] (MobileNetV2 convention) is a layer INSIDE the saved model.
PREPROCESSING_ID = "tf-resize224-bilinear-antialias__in-model-rescale-1to1__v1"
ARCHITECTURE_ID = "mobilenetv2_1.00_224__gap_dropout_dense_softmax__v1"

MODEL_FILENAME = "plant_disease_model.keras"
CLASS_NAMES_FILENAME = "class_names.json"
MODEL_INFO_FILENAME = "model_info.json"
INFO_SCHEMA_VERSION = 1

DEFAULT_SEED = 42
SPLIT_FRACTIONS = (0.70, 0.15, 0.15)  # train / validation / test
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

DATASET_NAME = "PlantVillage"
DATASET_REPO_URL = "https://github.com/spMohanty/PlantVillage-Dataset.git"

In [ ]:
%%writefile /content/pdc/src/preprocessing.py
"""Upload validation and image preprocessing.

The SAME resize operation (tf.image.resize, bilinear, antialias=True) is used for
training (``tf_load_image``) and for inference (``resize_for_model``). The rescale to
[-1, 1] lives inside the saved model, so the app never normalises by hand.
"""
from __future__ import annotations

import base64
import io
import logging
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps

from .constants import IMG_SIZE

log = logging.getLogger(__name__)

ALLOWED_EXTENSIONS = frozenset({"jpg", "jpeg", "png"})
ALLOWED_FORMATS = frozenset({"JPEG", "PNG"})
MIN_SIDE = 64
MAX_SIDE = 12000
MAX_PIXELS = 40_000_000
MIN_STD = 2.0  # reject (near-)uniform images


class ImageValidationError(ValueError):
    """Raised with a user-presentable message when an upload cannot be used."""


def check_extension(filename: str | None, allowed=ALLOWED_EXTENSIONS) -> str:
    ext = Path(filename or "").suffix.lower().lstrip(".")
    if ext not in allowed:
        raise ImageValidationError(
            "Unsupported file type. Please upload a JPG, JPEG or PNG image."
        )
    return ext


def decode_image_bytes(data: bytes, *, min_side=MIN_SIDE, max_side=MAX_SIDE,
                       max_pixels=MAX_PIXELS) -> Image.Image:
    """Decode untrusted bytes into an RGB PIL image or raise ImageValidationError."""
    if not data:
        raise ImageValidationError("The uploaded file is empty.")
    try:
        with Image.open(io.BytesIO(data)) as probe:
            fmt = probe.format
            width, height = probe.size
            if fmt not in ALLOWED_FORMATS:
                raise ImageValidationError(
                    "The file content is not a JPEG or PNG image (the extension does not match).")
            if width < min_side or height < min_side:
                raise ImageValidationError(
                    f"The image is too small ({width}x{height}). Minimum is {min_side}x{min_side} pixels.")
            if width > max_side or height > max_side or width * height > max_pixels:
                raise ImageValidationError(
                    f"The image is too large ({width}x{height}). Please upload a smaller photo.")
            probe.verify()  # structural check; the image must be re-opened afterwards
        with Image.open(io.BytesIO(data)) as img:
            img.load()
            img = ImageOps.exif_transpose(img)
            if img.mode in ("RGBA", "LA") or (img.mode == "P" and "transparency" in img.info):
                rgba = img.convert("RGBA")
                rgb = Image.new("RGB", rgba.size, (255, 255, 255))
                rgb.paste(rgba, mask=rgba.split()[-1])
            else:
                rgb = img.convert("RGB")
    except ImageValidationError:
        raise
    except Exception as exc:  # corrupted / truncated / decompression bomb / anything odd
        log.debug("Image decode failed: %r", exc)
        raise ImageValidationError(
            "The image could not be read. It may be corrupted or incomplete.") from exc

    small = np.asarray(rgb.resize((64, 64)), dtype=np.float32)
    if float(small.std()) < MIN_STD:
        raise ImageValidationError(
            "The image looks blank or uniform. Please upload a clear photo of a plant leaf.")
    return rgb


def pil_to_uint8(img: Image.Image) -> np.ndarray:
    return np.asarray(img.convert("RGB"), dtype=np.uint8)


def resize_for_model(rgb_uint8: np.ndarray) -> np.ndarray:
    """uint8 HxWx3 -> float32 (224,224,3) with values in 0..255 (inference path)."""
    import tensorflow as tf
    t = tf.convert_to_tensor(rgb_uint8, dtype=tf.uint8)
    t = tf.image.resize(t, (IMG_SIZE, IMG_SIZE), method="bilinear", antialias=True)
    return t.numpy().astype("float32")


def preprocess_pil_image(img: Image.Image) -> np.ndarray:
    """PIL image -> model input batch of shape (1, 224, 224, 3)."""
    return resize_for_model(pil_to_uint8(img))[None, ...]


def tf_load_image(path):
    """Training path: file path tensor -> float32 (224,224,3), same resize as inference."""
    import tensorflow as tf
    data = tf.io.read_file(path)
    img = tf.io.decode_image(data, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method="bilinear", antialias=True)
    return tf.cast(img, tf.float32)


def make_thumbnail_data_uri(img: Image.Image, max_side: int = 480) -> str:
    """Small JPEG data URI so the result page can show the upload without saving it."""
    thumb = img.copy()
    thumb.thumbnail((max_side, max_side))
    buf = io.BytesIO()
    thumb.save(buf, format="JPEG", quality=85)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode("ascii")

In [ ]:
%%writefile /content/pdc/src/model_builder.py
"""MobileNetV2 transfer-learning model (used by local training AND the Colab notebook)."""
from __future__ import annotations

from .constants import IMG_SHAPE


def build_model(num_classes: int, weights: str | None = "imagenet", dropout: float = 0.2,
                augment: bool = True):
    """Return an uncompiled Keras model.

    Input : float32 RGB, 0..255, shape (224, 224, 3)
    Layers: [training-only augmentation] -> Rescaling(-1..1) -> MobileNetV2 -> GAP
            -> Dropout -> Dense(softmax)
    Keras random-augmentation layers are inactive at inference, so the saved model is
    deterministic when used for prediction.
    """
    import keras
    from keras import layers

    inputs = keras.Input(shape=IMG_SHAPE, name="image_rgb_0_255")
    x = inputs
    if augment:
        x = layers.RandomFlip("horizontal_and_vertical", name="aug_flip")(x)
        x = layers.RandomRotation(0.1, name="aug_rotation")(x)
        x = layers.RandomZoom(0.1, name="aug_zoom")(x)
        x = layers.RandomContrast(0.1, name="aug_contrast")(x)
    x = layers.Rescaling(1.0 / 127.5, offset=-1.0, name="rescale_minus1_to_1")(x)
    base = keras.applications.MobileNetV2(
        input_shape=IMG_SHAPE, include_top=False, weights=weights, alpha=1.0,
        name="mobilenetv2_base")
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(dropout, name="dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32",
                           name="predictions")(x)
    return keras.Model(inputs, outputs, name="plant_disease_mobilenetv2")


def get_base(model):
    return model.get_layer("mobilenetv2_base")


def freeze_base(model) -> None:
    get_base(model).trainable = False


def unfreeze_top_layers(model, fine_tune_at: int = 100) -> int:
    """Unfreeze base layers from index ``fine_tune_at`` on (BatchNorm stays frozen).

    Returns the number of trainable base layers.
    """
    import keras
    base = get_base(model)
    base.trainable = True
    for i, layer in enumerate(base.layers):
        layer.trainable = i >= fine_tune_at and not isinstance(
            layer, keras.layers.BatchNormalization)
    return sum(1 for layer in base.layers if layer.trainable)


def compile_model(model, learning_rate: float) -> None:
    import keras
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

In [ ]:
%%writefile /content/pdc/src/dataset_utils.py
"""Dataset discovery, verification, leakage-aware splitting and tf.data pipelines."""
from __future__ import annotations

import hashlib
import json
import random
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd
from PIL import Image

from .constants import DEFAULT_SEED, IMAGE_EXTENSIONS, SPLIT_FRACTIONS

SKIP_DIR_NAMES = {"__macosx", ".git", ".ipynb_checkpoints"}
CONTAINER_DIR_NAMES = {"train", "valid", "val", "validation", "test"}


class DatasetError(RuntimeError):
    """Raised when the dataset is missing, malformed or unreadable."""


# --------------------------------------------------------------------------- discovery
def discover_images(root) -> pd.DataFrame:
    """Return a DataFrame(rel_path, class_name) for every image below ``root``.

    The class is the name of the image's parent folder. Folders such as train/valid/test
    are treated as containers, so the same class found in several of them is merged.
    """
    root = Path(root)
    if not root.is_dir():
        raise DatasetError(f"Dataset folder not found: {root}")
    rows = []
    for p in sorted(root.rglob("*")):
        if not p.is_file() or p.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        rel = p.relative_to(root)
        if any(part.lower() in SKIP_DIR_NAMES or part.startswith(".") for part in rel.parts[:-1]):
            continue
        parent = p.parent.name
        if p.parent == root or parent.lower() in CONTAINER_DIR_NAMES:
            continue  # image not inside a class folder
        rows.append((rel.as_posix(), parent))
    return pd.DataFrame(rows, columns=["rel_path", "class_name"])


def verify_dataset(root, check_readable: bool = True, min_classes: int = 2,
                   min_images_per_class: int = 10) -> dict:
    """Verify structure, counts and (optionally) that every image opens. Raises DatasetError."""
    df = discover_images(root)
    if df.empty:
        raise DatasetError(f"No class folders with JPG/PNG images were found under {root}.")
    counts = df["class_name"].value_counts().sort_index()
    if len(counts) < min_classes:
        raise DatasetError(f"Only {len(counts)} class folder(s) found; expected a folder per class.")
    unreadable = []
    if check_readable:
        for rel in df["rel_path"]:
            try:
                with Image.open(Path(root) / rel) as im:
                    im.verify()
            except Exception:
                unreadable.append(rel)
    small = counts[counts < min_images_per_class]
    report = {
        "root": str(root),
        "num_classes": int(len(counts)),
        "num_images": int(len(df)),
        "images_per_class": {k: int(v) for k, v in counts.items()},
        "min_images_in_a_class": int(counts.min()),
        "max_images_in_a_class": int(counts.max()),
        "readability_checked": bool(check_readable),
        "unreadable_files": unreadable,
        "classes_below_minimum": {k: int(v) for k, v in small.items()},
    }
    if unreadable:
        report["warning"] = (f"{len(unreadable)} unreadable image(s) were found and will be "
                             "excluded from training.")
    if len(small):
        raise DatasetError(f"Classes with fewer than {min_images_per_class} images: {list(small.index)}")
    return report


# --------------------------------------------------------------------------- grouping
_UUID_PREFIX = re.compile(r"^[0-9a-fA-F\-]{36}___")


def image_identifier(filename: str) -> str:
    """Original-photo identifier, following the PlantVillage repository's own convention."""
    name = filename.replace("_final_masked", "")
    if "___" in name:
        name = name.split("___")[-1]
    name = name.split("copy")[0]
    name = re.sub(r"\.(jpg|jpeg|png)$", "", name, flags=re.I)
    return name.strip().lower()


def load_leaf_map(path):
    if path and Path(path).is_file():
        with open(path, "r", encoding="utf-8") as fh:
            return json.load(fh)
    return None


def _file_md5(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


class _UnionFind:
    def __init__(self):
        self.parent = {}

    def find(self, x):
        self.parent.setdefault(x, x)
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[rb] = ra


def assign_groups(df: pd.DataFrame, root, leaf_map: dict | None = None,
                  hash_files: bool = True) -> tuple[pd.DataFrame, dict]:
    """Add a ``group`` column. Images that share a physical leaf (via leaf-map.json), an
    original-photo identifier, or identical bytes (MD5) get the same group and therefore
    always land in the same split."""
    uf = _UnionFind()
    info = {"leaf_map_used": leaf_map is not None, "leaf_map_hits": 0, "md5_hashing": hash_files}
    keys = []
    for rel, cls in zip(df["rel_path"], df["class_name"]):
        fname = Path(rel).name
        ident = image_identifier(fname)
        node = f"img::{rel}"
        uf.find(node)
        uf.union(node, f"id::{cls}::{ident}")
        if leaf_map:
            options = leaf_map.get(ident)
            if options:
                pick = next((o for o in options if cls in o), None) if len(options) > 1 else options[0]
                if pick:
                    uf.union(node, f"leaf::{pick}")
                    info["leaf_map_hits"] += 1
        keys.append(node)
    if hash_files:
        seen = {}
        for rel in df["rel_path"]:
            digest = _file_md5(Path(root) / rel)
            if digest in seen:
                uf.union(f"img::{rel}", f"img::{seen[digest]}")
            else:
                seen[digest] = rel
    out = df.copy()
    out["group"] = [uf.find(k) for k in keys]
    info["num_groups"] = int(out["group"].nunique())
    return out, info


# --------------------------------------------------------------------------- splitting
def subsample_per_class(df: pd.DataFrame, max_per_class: int, seed: int = DEFAULT_SEED) -> pd.DataFrame:
    parts = [g.sample(n=min(len(g), max_per_class), random_state=seed)
             for _, g in df.groupby("class_name")]
    return pd.concat(parts).sort_values("rel_path").reset_index(drop=True)


def make_splits(df: pd.DataFrame, fractions=SPLIT_FRACTIONS, seed: int = DEFAULT_SEED) -> pd.DataFrame:
    """Stratified, group-aware train/val/test split. Needs a ``group`` column."""
    rng = random.Random(seed)
    names = ["train", "val", "test"]
    split_of_group = {}
    for cls, g in df.groupby("class_name"):
        sizes = g.groupby("group").size()
        groups = sorted(sizes.index)
        if len(groups) < 3:
            raise DatasetError(f"Class '{cls}' has only {len(groups)} independent group(s); need >= 3.")
        rng.shuffle(groups)
        groups.sort(key=lambda k: -sizes[k])  # place big groups first, ties stay shuffled
        total = int(sizes.sum())
        targets = [total * f for f in fractions]
        assigned = [0, 0, 0]
        for k in groups:
            deficits = [(targets[i] - assigned[i]) / targets[i] for i in range(3)]
            i = max(range(3), key=lambda j: deficits[j])
            split_of_group[k] = names[i]
            assigned[i] += int(sizes[k])
    out = df.copy()
    out["split"] = out["group"].map(split_of_group)
    check_no_leakage(out)
    return out


def check_no_leakage(df: pd.DataFrame) -> None:
    per_group = df.groupby("group")["split"].nunique()
    bad = per_group[per_group > 1]
    if len(bad):
        raise DatasetError(f"Leakage: {len(bad)} group(s) appear in more than one split.")
    if df["split"].isna().any():
        raise DatasetError("Some images were not assigned to a split.")


# --------------------------------------------------------------------------- tf.data
def compute_class_weights(labels, num_classes: int) -> dict:
    counts = pd.Series(labels).value_counts()
    n = len(labels)
    return {int(c): float(n / (num_classes * counts[c])) for c in counts.index}


def build_tf_dataset(paths, labels, batch_size: int, training: bool, seed: int = DEFAULT_SEED):
    import tensorflow as tf
    from .preprocessing import tf_load_image
    ds = tf.data.Dataset.from_tensor_slices((list(paths), list(labels)))
    if training:
        ds = ds.shuffle(min(len(paths), 4096), seed=seed, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: (tf_load_image(p), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [ ]:
%%writefile /content/pdc/src/dataset_download.py
"""Automatic PlantVillage download (Method A).

Source : https://github.com/spMohanty/PlantVillage-Dataset (colour images in raw/color)
Paper  : Hughes & Salathe (2015), arXiv:1511.08060; Mohanty, Hughes & Salathe (2016),
         Frontiers in Plant Science 7:1419, doi:10.3389/fpls.2016.01419
Licence: the repository ships no LICENSE file. Treat the data as research material,
         cite the papers, and do NOT redistribute it inside your own ZIP.
"""
from __future__ import annotations

import os
import shutil
import subprocess
import tempfile
import urllib.request
import zipfile
from pathlib import Path

from .constants import DATASET_REPO_URL

ZIP_URL = "https://codeload.github.com/spMohanty/PlantVillage-Dataset/zip/refs/heads/master"
MANUAL_HELP = """\
Automatic download failed. Provide the dataset manually instead:
  1. Obtain PlantVillage (colour images, one folder per class, 38 classes), e.g. from
     https://github.com/spMohanty/PlantVillage-Dataset  (folder raw/color)
     or a Kaggle copy of PlantVillage.
  2. Then run ONE of:
       python training/prepare_dataset.py --source zip    --zip   path\\to\\dataset.zip
       python training/prepare_dataset.py --source folder --folder path\\to\\extracted\\folder
"""


class DatasetDownloadError(RuntimeError):
    pass


def _count_class_dirs(color_dir: Path) -> int:
    return sum(1 for d in color_dir.iterdir() if d.is_dir()) if color_dir.is_dir() else 0


def _download_with_git(dest: Path) -> Path:
    if shutil.which("git") is None:
        raise DatasetDownloadError("git is not installed.")
    repo = dest / "PlantVillage-Dataset"
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    if not repo.exists():
        cmd = ["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", DATASET_REPO_URL, str(repo)]
        r = subprocess.run(cmd, env=env, capture_output=True, text=True)
        if r.returncode != 0:
            raise DatasetDownloadError(f"git clone failed: {r.stderr.strip()[-300:]}")
    r = subprocess.run(["git", "-C", str(repo), "sparse-checkout", "set", "raw/color"],
                       env=env, capture_output=True, text=True)
    if r.returncode != 0:
        raise DatasetDownloadError(f"git sparse-checkout failed: {r.stderr.strip()[-300:]}")
    return repo


def _download_with_zip(dest: Path) -> Path:
    tmp = Path(tempfile.mkstemp(suffix=".zip", dir=dest)[1])
    try:
        print("Downloading repository ZIP (large, can take a while)...")
        with urllib.request.urlopen(ZIP_URL, timeout=60) as resp, open(tmp, "wb") as out:
            shutil.copyfileobj(resp, out, 1 << 20)
        if not zipfile.is_zipfile(tmp):
            raise DatasetDownloadError("The downloaded ZIP is incomplete or corrupted.")
        repo = dest / "PlantVillage-Dataset"
        with zipfile.ZipFile(tmp) as z:
            for m in z.namelist():
                parts = m.split("/", 1)
                if len(parts) < 2:
                    continue
                rel = parts[1]
                if not (rel.startswith("raw/color/") or rel == "leaf-map.json"):
                    continue
                target = (repo / rel).resolve()
                if not str(target).startswith(str(repo.resolve())):
                    raise DatasetDownloadError("Unsafe path in ZIP.")
                if m.endswith("/"):
                    target.mkdir(parents=True, exist_ok=True)
                else:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with z.open(m) as src, open(target, "wb") as dst:
                        shutil.copyfileobj(src, dst)
        return repo
    finally:
        tmp.unlink(missing_ok=True)


def download_plantvillage(dest) -> tuple[Path, Path | None]:
    """Download PlantVillage colour images. Returns (class_root, leaf_map_path_or_None).

    Never assumes success: the result is checked for class folders before returning.
    """
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    errors = []
    for name, fn in (("git sparse checkout", _download_with_git), ("ZIP download", _download_with_zip)):
        try:
            repo = fn(dest)
            color = repo / "raw" / "color"
            n = _count_class_dirs(color)
            if n < 30:
                raise DatasetDownloadError(f"Expected ~38 class folders, found {n}.")
            leaf_map = repo / "leaf-map.json"
            return color, (leaf_map if leaf_map.is_file() else None)
        except Exception as exc:  # try the next method
            errors.append(f"{name}: {exc}")
            print(f"[{name}] failed: {exc}")
    raise DatasetDownloadError("; ".join(errors) + "\n" + MANUAL_HELP)

In [ ]:
%%writefile /content/pdc/training/prepare_dataset.py
"""Prepare and VERIFY the PlantVillage dataset (never assumes a download succeeded).

  python training/prepare_dataset.py --source auto
  python training/prepare_dataset.py --source zip    --zip    C:\\path\\to\\dataset.zip
  python training/prepare_dataset.py --source folder --folder C:\\path\\to\\extracted_folder

Writes dataset/dataset_info.json and, under results/: dataset_stats.json,
dataset_class_distribution.png and dataset_samples.png (real sample images).
"""
from __future__ import annotations

import argparse
import json
import random
import shutil
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

from src.constants import DATASET_NAME, IMAGE_EXTENSIONS  # noqa: E402
from src.dataset_download import DatasetDownloadError, MANUAL_HELP, download_plantvillage  # noqa: E402
from src.dataset_utils import (CONTAINER_DIR_NAMES, SKIP_DIR_NAMES, DatasetError,  # noqa: E402
                               discover_images, verify_dataset)

CITATION = ("Hughes & Salathe (2015) arXiv:1511.08060; Mohanty, Hughes & Salathe (2016) "
            "Frontiers in Plant Science 7:1419, doi:10.3389/fpls.2016.01419")


def _has_images(d: Path) -> bool:
    return any(p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS for p in d.iterdir())


def find_class_root(folder: Path, max_depth: int = 4) -> Path:
    """Locate the folder that directly holds one sub-folder per class."""
    queue = [(Path(folder), 0)]
    while queue:
        d, depth = queue.pop(0)
        subs = [s for s in d.iterdir() if s.is_dir() and not s.name.startswith(".")
                and s.name.lower() not in SKIP_DIR_NAMES]
        if len([s for s in subs if _has_images(s)]) >= 2:
            return d
        if any(s.name.lower() in CONTAINER_DIR_NAMES for s in subs):
            return d
        if depth < max_depth:
            queue.extend((s, depth + 1) for s in subs)
    raise DatasetError(f"Could not find class folders (one folder per class, containing images) in {folder}.")


def safe_extract(zip_path: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    base = dest.resolve()
    with zipfile.ZipFile(zip_path) as z:
        bad = z.testzip()
        if bad:
            raise DatasetError(f"The ZIP is corrupted (first bad member: {bad}).")
        for m in z.infolist():
            target = (dest / m.filename).resolve()
            if not str(target).startswith(str(base)):
                raise DatasetError("The ZIP contains an unsafe path and was not extracted.")
        z.extractall(dest)


def _rel_or_abs(path: Path) -> str:
    try:
        return path.resolve().relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return str(path.resolve())


def resolve_dataset_location(project_root: Path, data_dir: str | None, leaf_map: str | None):
    """Return (dataset_root, leaf_map_path_or_None) from CLI args or dataset/dataset_info.json."""
    info_path = project_root / "dataset" / "dataset_info.json"
    info = json.loads(info_path.read_text(encoding="utf-8")) if info_path.is_file() else {}
    if data_dir:
        root = Path(data_dir)
    elif info.get("dataset_root"):
        root = Path(info["dataset_root"])
        if not root.is_absolute():
            root = project_root / root
    else:
        raise DatasetError("No dataset prepared yet. Run: python training/prepare_dataset.py --source auto")
    lm = leaf_map or info.get("leaf_map")
    lm_path = Path(lm) if lm else None
    if lm_path and not lm_path.is_absolute():
        lm_path = project_root / lm_path
    return root, (lm_path if lm_path and lm_path.is_file() else None)


def write_figures(root: Path, report: dict, results_dir: Path, seed: int = 42) -> None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from PIL import Image
    counts = report["images_per_class"]
    names = list(counts)
    fig, ax = plt.subplots(figsize=(10, max(6, len(names) * 0.25)))
    ax.barh(names, [counts[n] for n in names], color="#40916c")
    ax.set_xlabel("Number of images")
    ax.set_title(f"{DATASET_NAME}: images per class ({report['num_images']} images, {report['num_classes']} classes)")
    ax.tick_params(axis="y", labelsize=7)
    fig.tight_layout()
    fig.savefig(results_dir / "dataset_class_distribution.png", dpi=110)
    plt.close(fig)

    df = discover_images(root)
    rng = random.Random(seed)
    classes = sorted(df["class_name"].unique())
    step = max(1, len(classes) // 24)
    chosen = classes[::step][:24]
    cols = 6
    rows = (len(chosen) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.4, rows * 2.6))
    for ax in axes.ravel():
        ax.axis("off")
    for ax, cls in zip(axes.ravel(), chosen):
        rel = rng.choice(list(df.loc[df["class_name"] == cls, "rel_path"]))
        with Image.open(root / rel) as im:
            ax.imshow(im.convert("RGB"))
        ax.set_title(cls.replace("___", "\n").replace("_", " ")[:34], fontsize=6)
    fig.suptitle(f"Sample images from the dataset ({len(chosen)} of {len(classes)} classes)")
    fig.tight_layout()
    fig.savefig(results_dir / "dataset_samples.png", dpi=110)
    plt.close(fig)


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--source", choices=["auto", "zip", "folder"], required=True)
    ap.add_argument("--zip", help="path to a dataset ZIP (with --source zip)")
    ap.add_argument("--folder", help="path to an extracted dataset folder (with --source folder)")
    ap.add_argument("--leaf-map", help="optional leaf-map.json (leaf-level grouping to avoid split leakage)")
    ap.add_argument("--download-dir", default=str(PROJECT_ROOT / "dataset" / "_download"))
    ap.add_argument("--results-dir", default=str(PROJECT_ROOT / "results"))
    ap.add_argument("--skip-readability-check", action="store_true")
    args = ap.parse_args(argv)

    results_dir = Path(args.results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)
    leaf_map = Path(args.leaf_map) if args.leaf_map else None
    try:
        if args.source == "auto":
            root, found_map = download_plantvillage(args.download_dir)
            leaf_map = leaf_map or found_map
        elif args.source == "zip":
            if not args.zip or not Path(args.zip).is_file():
                raise DatasetError("Please pass --zip PATH to an existing ZIP file.")
            dest = PROJECT_ROOT / "dataset" / "_extracted" / Path(args.zip).stem
            print(f"Extracting {args.zip} ...")
            safe_extract(Path(args.zip), dest)
            root = find_class_root(dest)
            leaf_map = leaf_map or next(iter(dest.rglob("leaf-map.json")), None)
        else:
            if not args.folder or not Path(args.folder).is_dir():
                raise DatasetError("Please pass --folder PATH to an existing folder.")
            root = find_class_root(Path(args.folder))
            leaf_map = leaf_map or next(iter(Path(args.folder).rglob("leaf-map.json")), None)
        print(f"Dataset root: {root}\nVerifying structure, counts and image readability ...")
        report = verify_dataset(root, check_readable=not args.skip_readability_check)
    except (DatasetError, DatasetDownloadError) as exc:
        print(f"\nERROR: {exc}\n\n{MANUAL_HELP}", file=sys.stderr)
        return 1

    stats = dict(report, dataset=DATASET_NAME, source_method=args.source, citation=CITATION,
                 verified_at=datetime.now(timezone.utc).isoformat(timespec="seconds"),
                 leaf_map_available=bool(leaf_map))
    (results_dir / "dataset_stats.json").write_text(json.dumps(stats, indent=2), encoding="utf-8")
    info = {"dataset_root": _rel_or_abs(Path(root)), "leaf_map": _rel_or_abs(Path(leaf_map)) if leaf_map else None,
            "method": args.source, "verified_at": stats["verified_at"],
            "num_classes": report["num_classes"], "num_images": report["num_images"]}
    (PROJECT_ROOT / "dataset").mkdir(exist_ok=True)
    (PROJECT_ROOT / "dataset" / "dataset_info.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
    write_figures(Path(root), report, results_dir)
    print(f"\nVerified {report['num_images']} images in {report['num_classes']} classes "
          f"(per-class min {report['min_images_in_a_class']}, max {report['max_images_in_a_class']}).")
    if report["unreadable_files"]:
        print(f"WARNING: {len(report['unreadable_files'])} unreadable file(s) will be excluded.")
    if not leaf_map:
        print("NOTE: leaf-map.json not found; splits will group by photo id and identical files only "
              "(see README, 'Data leakage').")
    print("Wrote dataset/dataset_info.json and results/dataset_*.{json,png}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile /content/pdc/training/evaluate.py
"""Evaluate the trained model on the held-out TEST split and write figures/metrics.

  python training/evaluate.py          (uses models/, results/split_manifest.csv)
"""
from __future__ import annotations

import argparse
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

from src.constants import CLASS_NAMES_FILENAME, MODEL_FILENAME, MODEL_INFO_FILENAME  # noqa: E402
from src.dataset_utils import build_tf_dataset  # noqa: E402


def evaluate_model(model, test_df: pd.DataFrame, data_root: Path, class_names: list,
                   results_dir: Path, batch_size: int = 32) -> dict:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                                 precision_recall_fscore_support)
    idx = {c: i for i, c in enumerate(class_names)}
    y_true = test_df["class_name"].map(idx).to_numpy()
    ds = build_tf_dataset([str(data_root / p) for p in test_df["rel_path"]], y_true, batch_size, training=False)
    probs = model.predict(ds, verbose=0)
    y_pred = probs.argmax(axis=1)
    k = min(3, len(class_names))
    topk = np.argsort(probs, axis=1)[:, -k:]
    labels = list(range(len(class_names)))
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    _, _, wf1, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average="weighted", zero_division=0)
    report = classification_report(y_true, y_pred, labels=labels, target_names=class_names,
                                   zero_division=0, output_dict=True)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    results_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(report).T.to_csv(results_dir / "classification_report.csv")
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(results_dir / "confusion_matrix.csv")

    norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(14, 12))
    im = ax.imshow(norm, cmap="Greens", vmin=0, vmax=1)
    ax.set_xticks(labels)
    ax.set_yticks(labels)
    short = [c.replace("___", ": ").replace("_", " ")[:30] for c in class_names]
    ax.set_xticklabels(short, rotation=90, fontsize=6)
    ax.set_yticklabels(short, fontsize=6)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title("Confusion matrix on the test split (row-normalised)")
    fig.colorbar(im, ax=ax, fraction=0.03)
    fig.tight_layout()
    fig.savefig(results_dir / "confusion_matrix.png", dpi=110)
    plt.close(fig)

    summary = {
        "evaluated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "split": "test", "num_test_images": int(len(test_df)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "top3_accuracy": float(np.mean([y_true[i] in topk[i] for i in range(len(y_true))])),
        "macro_precision": float(p), "macro_recall": float(r), "macro_f1": float(f1),
        "weighted_f1": float(wf1), "per_class": report,
    }
    (results_dir / "evaluation_results.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return summary


def plot_training_curves(history_df: pd.DataFrame, path: Path) -> None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    x = np.arange(1, len(history_df) + 1)
    for ax, key, title in zip(axes, ["accuracy", "loss"], ["Accuracy", "Loss"]):
        ax.plot(x, history_df[key], label="train")
        ax.plot(x, history_df["val_" + key], label="validation")
        if "stage" in history_df and history_df["stage"].nunique() > 1:
            first_ft = int(np.argmax(history_df["stage"].to_numpy() == 2)) + 1
            ax.axvline(first_ft - 0.5, color="grey", linestyle="--", label="fine-tuning starts")
        ax.set_xlabel("Epoch")
        ax.set_title(title)
        ax.legend()
        ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)


def main(argv=None) -> int:
    from training.prepare_dataset import resolve_dataset_location
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--model-dir", default=str(PROJECT_ROOT / "models"))
    ap.add_argument("--results-dir", default=str(PROJECT_ROOT / "results"))
    ap.add_argument("--data-dir", default=None)
    ap.add_argument("--batch-size", type=int, default=32)
    args = ap.parse_args(argv)
    model_dir, results_dir = Path(args.model_dir), Path(args.results_dir)
    manifest = results_dir / "split_manifest.csv"
    if not (model_dir / MODEL_FILENAME).is_file() or not manifest.is_file():
        print("ERROR: trained model or results/split_manifest.csv not found. Train first (training/train.py).",
              file=sys.stderr)
        return 1
    import keras
    class_names = json.loads((model_dir / CLASS_NAMES_FILENAME).read_text(encoding="utf-8"))
    model = keras.models.load_model(model_dir / MODEL_FILENAME, compile=False)
    df = pd.read_csv(manifest)
    test_df = df[df["split"] == "test"]
    root, _ = resolve_dataset_location(PROJECT_ROOT, args.data_dir, None)
    summary = evaluate_model(model, test_df, root, class_names, results_dir, args.batch_size)
    info_path = model_dir / MODEL_INFO_FILENAME
    if info_path.is_file():
        info = json.loads(info_path.read_text(encoding="utf-8"))
        info["evaluation"] = {k: v for k, v in summary.items() if k != "per_class"}
        info_path.write_text(json.dumps(info, indent=2), encoding="utf-8")
    print(f"Test accuracy {summary['accuracy']:.4f} | top-3 {summary['top3_accuracy']:.4f} | "
          f"macro-F1 {summary['macro_f1']:.4f} on {summary['num_test_images']} images")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile /content/pdc/training/train.py
"""Train the MobileNetV2 plant-disease classifier (identical code path locally and on Colab).

  python training/train.py --epochs 10 --fine-tune-epochs 5

Outputs (written to --model-dir only after training AND evaluation finished successfully):
  plant_disease_model.keras, class_names.json, model_info.json
and to --results-dir: split_manifest.csv, training_history.csv/json, training_curves.png,
evaluation_results.json, classification_report.csv, confusion_matrix.csv/png.
"""
from __future__ import annotations

import argparse
import json
import os
import platform
import shutil
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

from src.constants import (ARCHITECTURE_ID, CLASS_NAMES_FILENAME, DATASET_NAME, DEFAULT_SEED,  # noqa: E402
                           IMG_SIZE, INFO_SCHEMA_VERSION, MODEL_FILENAME, MODEL_INFO_FILENAME,
                           PREPROCESSING_ID, SPLIT_FRACTIONS)
from src.dataset_utils import (DatasetError, assign_groups, build_tf_dataset,  # noqa: E402
                               compute_class_weights, load_leaf_map, make_splits,
                               subsample_per_class, verify_dataset)
from src.model_builder import (build_model, compile_model, freeze_base,  # noqa: E402
                               unfreeze_top_layers)
from training.evaluate import evaluate_model, plot_training_curves  # noqa: E402
from training.prepare_dataset import resolve_dataset_location  # noqa: E402


def parse_args(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data-dir", default=None, help="class-folder root (default: from dataset/dataset_info.json)")
    ap.add_argument("--leaf-map", default=None)
    ap.add_argument("--model-dir", default=str(PROJECT_ROOT / "models"))
    ap.add_argument("--results-dir", default=str(PROJECT_ROOT / "results"))
    ap.add_argument("--epochs", type=int, default=10, help="max epochs for the head-training stage")
    ap.add_argument("--fine-tune-epochs", type=int, default=0, help="max epochs for optional fine-tuning (0 = off)")
    ap.add_argument("--fine-tune-at", type=int, default=100, help="first MobileNetV2 layer index to unfreeze")
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--fine-tune-lr", type=float, default=1e-5)
    ap.add_argument("--dropout", type=float, default=0.2)
    ap.add_argument("--patience", type=int, default=3)
    ap.add_argument("--weights", choices=["imagenet", "none"], default="imagenet",
                    help="'none' = random init (NOT transfer learning); only for pipeline checks")
    ap.add_argument("--max-per-class", type=int, default=None,
                    help="use only N images per class (pipeline check ONLY; recorded in model_info.json)")
    ap.add_argument("--no-class-weights", action="store_true")
    ap.add_argument("--skip-hash", action="store_true", help="skip MD5 duplicate detection")
    ap.add_argument("--skip-readability-check", action="store_true")
    ap.add_argument("--seed", type=int, default=DEFAULT_SEED)
    return ap.parse_args(argv)


def main(argv=None) -> int:
    args = parse_args(argv)
    t0 = time.time()
    import keras
    import tensorflow as tf
    keras.utils.set_random_seed(args.seed)

    gpus = tf.config.list_physical_devices("GPU")
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    device = f"GPU ({len(gpus)} device(s))" if gpus else "CPU"
    print(f"TensorFlow {tf.__version__} | Keras {keras.__version__} | training on {device}")
    if not gpus:
        print("No GPU detected: training on CPU is slow for the full dataset. Consider Google Colab (see README).")

    model_dir, results_dir = Path(args.model_dir), Path(args.results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)
    tmp = model_dir / "_training_tmp"
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True, exist_ok=True)

    # ---- dataset ---------------------------------------------------------------
    try:
        root, leaf_map_path = resolve_dataset_location(PROJECT_ROOT, args.data_dir, args.leaf_map)
        print(f"Dataset root: {root}")
        report = verify_dataset(root, check_readable=not args.skip_readability_check)
    except DatasetError as exc:
        print(f"ERROR: {exc}", file=sys.stderr)
        return 1
    from src.dataset_utils import discover_images
    df = discover_images(root)
    df = df[~df["rel_path"].isin(report["unreadable_files"])].reset_index(drop=True)
    if args.max_per_class:
        df = subsample_per_class(df, args.max_per_class, args.seed)
        print(f"WARNING: SUBSET MODE ({args.max_per_class} images/class). Pipeline check only; not a real model.")
    leaf_map = load_leaf_map(leaf_map_path)
    print("Grouping images (leaf map / photo id / identical files) to prevent split leakage ...")
    df, group_info = assign_groups(df, root, leaf_map, hash_files=not args.skip_hash)
    df = make_splits(df, SPLIT_FRACTIONS, args.seed)
    class_names = sorted(df["class_name"].unique())
    idx = {c: i for i, c in enumerate(class_names)}
    df["label"] = df["class_name"].map(idx)
    df[["rel_path", "class_name", "group", "split"]].to_csv(results_dir / "split_manifest.csv", index=False)
    split_sizes = {s: int((df["split"] == s).sum()) for s in ("train", "val", "test")}
    print(f"{len(class_names)} classes | split sizes {split_sizes} | groups {group_info['num_groups']} "
          f"| leaf-map hits {group_info['leaf_map_hits']}")

    def ds_for(split, training):
        d = df[df["split"] == split]
        return build_tf_dataset([str(root / p) for p in d["rel_path"]], d["label"].to_numpy(),
                                args.batch_size, training, args.seed)

    train_ds, val_ds = ds_for("train", True), ds_for("val", False)
    class_weight = None if args.no_class_weights else compute_class_weights(
        df.loc[df["split"] == "train", "label"].to_numpy(), len(class_names))

    # ---- model -----------------------------------------------------------------
    pretrained = args.weights == "imagenet"
    try:
        model = build_model(len(class_names), weights="imagenet" if pretrained else None, dropout=args.dropout)
    except Exception as exc:
        print(f"ERROR: could not build the model with ImageNet weights ({exc}).\n"
              "Check your internet connection. Not falling back to random weights silently; "
              "use --weights none only for a deliberate pipeline check.", file=sys.stderr)
        return 1
    if pretrained:
        freeze_base(model)
    compile_model(model, args.lr)
    print(f"Parameters: {model.count_params():,} total | stage 1 trainable: "
          f"{sum(int(keras.ops.size(w)) for w in model.trainable_weights):,}")

    history_rows, stages = [], []
    best = {"val_loss": float("inf"), "stage": None}

    def run_stage(stage, epochs):
        cbs = [
            keras.callbacks.EarlyStopping(monitor="val_loss", patience=args.patience, restore_best_weights=True),
            keras.callbacks.ModelCheckpoint(str(tmp / f"checkpoint_stage{stage}.keras"), monitor="val_loss",
                                            save_best_only=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=max(1, args.patience - 1)),
        ]
        h = model.fit(train_ds, validation_data=val_ds, epochs=epochs, class_weight=class_weight,
                      callbacks=cbs, verbose=2).history
        n = len(h["loss"])
        for e in range(n):
            history_rows.append({"stage": stage, "epoch_in_stage": e + 1, **{k: float(v[e]) for k, v in h.items()}})
        val_loss, val_acc = model.evaluate(val_ds, verbose=0)
        stages.append({"stage": stage, "epochs_run": n, "val_loss_after_restore": float(val_loss),
                       "val_accuracy_after_restore": float(val_acc)})
        if val_loss < best["val_loss"]:
            best.update(val_loss=float(val_loss), stage=stage)
            model.save(tmp / MODEL_FILENAME)
            print(f"Stage {stage}: new best validation loss {val_loss:.4f} -> saved")

    run_stage(1, args.epochs)
    if args.fine_tune_epochs > 0:
        if not pretrained:
            print("Skipping fine-tuning: the base was trained from scratch (--weights none).")
        else:
            n_train = unfreeze_top_layers(model, args.fine_tune_at)
            compile_model(model, args.fine_tune_lr)
            print(f"Fine-tuning {n_train} base layers (lr={args.fine_tune_lr}).")
            run_stage(2, args.fine_tune_epochs)

    # ---- evaluate the SAVED artefact on the untouched test split ----------------
    hist_df = pd.DataFrame(history_rows)
    hist_df.to_csv(results_dir / "training_history.csv", index=False)
    (results_dir / "training_history.json").write_text(hist_df.to_json(orient="records", indent=2), encoding="utf-8")
    plot_training_curves(hist_df, results_dir / "training_curves.png")
    saved = keras.models.load_model(tmp / MODEL_FILENAME, compile=False)
    test_df = df[df["split"] == "test"]
    summary = evaluate_model(saved, test_df, root, class_names, results_dir, args.batch_size)
    print(f"TEST accuracy {summary['accuracy']:.4f} | top-3 {summary['top3_accuracy']:.4f} | "
          f"macro-F1 {summary['macro_f1']:.4f} ({summary['num_test_images']} images)")

    # ---- publish model files (only now: training finished and was evaluated) ----
    stamp = datetime.now(timezone.utc)
    info = {
        "schema_version": INFO_SCHEMA_VERSION, "is_test_fixture": False,
        "model_version": "pdc-mobilenetv2-" + stamp.strftime("%Y%m%d-%H%M%S"),
        "trained_at": stamp.isoformat(timespec="seconds"),
        "architecture_id": ARCHITECTURE_ID, "preprocessing_id": PREPROCESSING_ID, "img_size": IMG_SIZE,
        "num_classes": len(class_names), "class_names": class_names,
        "pretrained_weights": "imagenet" if pretrained else "none",
        "training_mode": "subset" if args.max_per_class else "full", "max_per_class": args.max_per_class,
        "dataset": {"name": DATASET_NAME, "num_images_used": int(len(df)), "num_images_found": report["num_images"],
                    "unreadable_excluded": len(report["unreadable_files"]),
                    "grouping": group_info, "split_fractions": list(SPLIT_FRACTIONS)},
        "split_sizes": split_sizes,
        "config": {k: v for k, v in vars(args).items() if k not in ("data_dir", "model_dir", "results_dir", "leaf_map")},
        "stages": stages, "best_stage": best["stage"], "best_val_loss": best["val_loss"],
        "device": device, "python_version": platform.python_version(),
        "tensorflow_version": tf.__version__, "keras_version": keras.__version__,
        "training_seconds": round(time.time() - t0, 1),
        "evaluation": {k: v for k, v in summary.items() if k != "per_class"},
    }
    shutil.copy2(tmp / MODEL_FILENAME, model_dir / (MODEL_FILENAME + ".part"))
    os.replace(model_dir / (MODEL_FILENAME + ".part"), model_dir / MODEL_FILENAME)
    (model_dir / CLASS_NAMES_FILENAME).write_text(json.dumps(class_names, indent=2), encoding="utf-8")
    (model_dir / MODEL_INFO_FILENAME).write_text(json.dumps(info, indent=2), encoding="utf-8")
    shutil.rmtree(tmp, ignore_errors=True)
    print(f"Saved {MODEL_FILENAME}, {CLASS_NAMES_FILENAME}, {MODEL_INFO_FILENAME} to {model_dir}")
    print("Training finished successfully.")
    return 0


if __name__ == "__main__":
    sys.exit(main())

## 3. Dataset: download (or load from Drive) and verify

In [ ]:
args = [sys.executable, "training/prepare_dataset.py", "--results-dir", OUT + "/results",
        "--download-dir", "/content/dataset_download"]
if DATASET_SOURCE == "auto":
    args += ["--source", "auto"]
else:
    args += ["--source", "zip", "--zip", DRIVE_ZIP]
run(args)

In [ ]:
from IPython.display import Image, display
display(Image(OUT + "/results/dataset_samples.png"))
display(Image(OUT + "/results/dataset_class_distribution.png"))

## 4. Train (MobileNetV2 transfer learning, early stopping, checkpointing, optional fine-tuning)

In [ ]:
run([sys.executable, "training/train.py",
     "--epochs", str(EPOCHS), "--fine-tune-epochs", str(FINE_TUNE_EPOCHS), "--batch-size", str(BATCH_SIZE),
     "--model-dir", OUT + "/models", "--results-dir", OUT + "/results"])

## 5. Results (computed on the held-out test split by the training script)

In [ ]:
import json
res = json.load(open(OUT + "/results/evaluation_results.json"))
print({k: res[k] for k in ("num_test_images", "accuracy", "top3_accuracy", "macro_f1", "weighted_f1")})
display(Image(OUT + "/results/training_curves.png"))
display(Image(OUT + "/results/confusion_matrix.png"))

## 6. Download the model files

In [ ]:
import shutil
print(open(OUT + "/models/model_info.json").read()[:600], "...")
archive = shutil.make_archive("/content/plant_disease_model_files", "zip", OUT + "/models")
if USE_DRIVE:
    shutil.copytree(OUT, DRIVE_DIR + "/output", dirs_exist_ok=True)
    print("Copied to Google Drive:", DRIVE_DIR + "/output")
from google.colab import files
files.download(archive)

## 7. Use the model in the web application
Unzip the download and copy these three files into the project's `models/` folder (same folder as `.gitkeep`):

```
models/plant_disease_model.keras
models/class_names.json
models/model_info.json
```

Optionally copy the figures from `output/results/` (`training_curves.png`, `confusion_matrix.png`, `evaluation_results.json`, ...) into the project's `results/` folder so the About page can show them. Then start the app (`scripts\run_windows.bat`). The app runs a compatibility check on load and refuses incompatible files; it never substitutes an untrained model.